[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-getting-started/blob/main/notebooks/05_workflows.ipynb)

# 05 · Workflows — reproducible, templated research

**Workflows** turn a research process into a reusable **template** you can run
again and again — over different companies, dates, or universes — and get a
consistent, structured report each time. It's the general-purpose engine behind
deep-research reports on **[Bigdata.com](https://bigdata.com)**.

**What it's good for**
- Standardised analyses you re-run regularly (e.g. a weekly company brief)
- Running the same prompt across a whole watchlist or universe
- Going from a saved template → a single API call in your own systems

`POST /v1/workflow/execute` (on **`agents.bigdata.com`**) streams results as SSE,
just like the Research Agent. You can run an **inline** template or a **stored
template by ID**.

In [ ]:
!pip install -q requests env-colab-pass

In [ ]:
import requests
from env_colab_pass import passutil

# Your API key is read from the BIGDATA_API_KEY environment variable.
# Never hard-code keys in a notebook you plan to share or commit.
API_KEY = passutil.get_secret_value("BIGDATA_API_KEY")

# Bigdata exposes two hosts:
#   - api.bigdata.com    -> Knowledge Graph & Search Service (deterministic APIs)
#   - agents.bigdata.com -> Research Agent & Workflows (AI agents, streamed)
API_BASE = "https://api.bigdata.com"
AGENTS_BASE = "https://agents.bigdata.com"

# The key must be sent in the X-API-KEY header on every request.
HEADERS = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

### SSE helper
Same streaming idea as notebook 04; here the event payload is under `delta`.

In [ ]:
import json

def run_workflow(template, inputs=None, model_name="base", time_range=None, model="base", show_progress=True):
    """Execute a workflow. `template` is an inline dict or a stored template ID (str)."""
    payload = {"template": template, "model_name": model_name}
    if inputs:
        payload["input"] = inputs
    if time_range:
        payload["time_range"] = time_range

    answer_parts = []
    with requests.post(f"{AGENTS_BASE}/v1/workflow/execute", headers=HEADERS, json=payload,
                       stream=True, timeout=600) as r:
        r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            event = json.loads(line[len("data:"):].strip())
            delta = event.get("delta", {})
            mtype = delta.get("type")
            if mtype == "ERROR":
                raise RuntimeError(delta.get("error"))
            if mtype == "ACTION" and show_progress:
                print("  action:", delta.get("tool_name"))
            elif mtype == "ANSWER":
                answer_parts.append(delta.get("content", ""))
    return "".join(answer_parts)

### Example 1 — Run an inline template
Define a template with a `{{ placeholder }}` in the prompt and declare it in
`expected_input`. Supply values via `input`. (Declaring `expected_input` is
required whenever you pass `input`.)

In [ ]:
template = {
    "name": "Company Brief",
    "prompt": 
        """
        Build an institutional initiation-grade brief on {{ company }} that a PM can read cold and walk into an idea-review meeting with.

        The objective is not a description of the business — it is a decision-useful read on earnings power, capital returns, competitive position, and where the current price embeds expectations the market may have wrong.

        Quantify everything: growth rates, margins, returns on capital, leverage, and current multiples against the company’s own history and against peers. Wherever you cite a number, anchor it to a period and a source.

        Flag the two or three debates where the bull and bear views actually diverge, and frame the open questions a deeper diligence pass would need to resolve.

        Write in the terse, quantitative voice of a buy-side note — no filler, no boilerplate. Where a data point is not disclosed, write "N/A — not disclosed" rather than estimating.

        Deliver the final report in the exact structure and tables defined by the output schema. Use well-formed markdown tables, populate every cell, and where an item is not disclosed write "N/A — not disclosed" rather than guessing. Do not restate the section/table layout in this prompt.

        This is a written research note, not a data sheet: open with the framing paragraph the output schema defines, follow every table with its "Read:" interpretation, and close with the synthesis section. Interpretation is the deliverable — state what the evidence means, not only what it is, and keep every claim evidence-bound.
   
        """
    ,
    "expected_input": {"company": {"type": "string"}},
        
    "research_plan": {
        "title": "Research plan",
        "steps": [
            {
                "description": (
                    "Extract the business model and segment structure from the most recent annual report: "
                    "primary products/services, revenue by reporting segment (latest FY, % of total), revenue by geography "
                    "(% of total), and the principal end markets and demand drivers. "
                    "Deliver a 2-3 paragraph overview with the segment and geographic mix tables."
                )
            },
            {
                "description": (
                    "Extract revenue, operating income, net income, and operating cash flow for the last 3 fiscal years. "
                    "Compute YoY growth and the 2-year revenue CAGR, and compute operating margin for each year. "
                    "State whether growth is accelerating or decelerating and whether margins are expanding or compressing, with the primary driver."
                )
            },
            {
                "description": (
                    "Compute free cash flow (operating cash flow minus capex) for the last 3 fiscal years and the FCF/net income conversion ratio for each year. "
                    "Compute ROIC for the last 2 full years (NOPAT over invested capital) and net debt/EBITDA for the last 2 full years. "
                    "Note whether ROIC clears a 8-12% cost-of-capital range and whether the spread is widening or compressing."
                )
            },
            {
                "description": (
                    "Identify the 2-3 closest competitors named in the annual report's competition discussion and in industry coverage. "
                    "For each, extract latest-FY revenue and revenue growth, and assess relative scale versus the company (e.g., #1/#2 by revenue). "
                    "Note the company's stated source of competitive advantage or disadvantage in one line."
                )
            },
            {
                "description": (
                    "Extract the senior management team from the most recent proxy statement (CEO, CFO, and one or two other key executives) "
                    "with title and years in current role. Report combined insider ownership as a percentage of shares outstanding, "
                    "and flag any executive turnover in the past 2 years or unusually short CEO/CFO tenure."
                )
            },
            {
                "description": (
                    "Review the two most recent earnings call transcripts for management's stated strategic priorities, "
                    "capital-allocation intentions, and margin levers. Identify the 2-3 themes management is emphasizing and, "
                    "for each, note whether the prior track record supports the credibility of the claim."
                )
            },
            {
                "description": (
                    "Extract current valuation multiples — NTM P/E, EV/EBITDA, P/S, and FCF yield. "
                    "From sell-side research and equity notes, capture the company's own 5-year average or historical trading range for each multiple. "
                    "State whether the stock trades at a premium or discount to its own history and what that premium/discount implies the market is pricing."
                )
            },
            {
                "description": (
                    "For the 2-3 competitors identified earlier, extract current P/E and EV/EBITDA and compute a peer median for each. "
                    "Compare the company's current multiples to the peer median and quantify the premium or discount, "
                    "noting whether relative valuation is justified by superior growth or returns."
                )
            },
            {
                "description": (
                    "Survey company press releases, investor presentations, and SEC filings (8-K, prospectuses) from the last 6 months, "
                    "plus recent news, for dated catalysts over the next 6-12 months: earnings dates, product launches, regulatory decisions, "
                    "capital-return actions, and strategic events. For each, note timing and the likely direction of impact on the stock."
                )
            }
        ]
    }    
}

answer = run_workflow(template, inputs={"company": "Apple"})
print("\n=== RESULT ===\n")
print(answer)

### Example 2 — Create and re-use workflow template
Create due diligence template 

In [ ]:
# Define a due diligence template
due_diligence_template = {
    "name": "Demo - Company Due Diligence Report",
    "description": "Comprehensive due diligence analysis for investment research",
    "prompt": """
Perform comprehensive due diligence on {{ company }}.

Your analysis should include:

1. **Business Overview** - Core business model, competitive positioning
2. **Recent Developments** - Latest news, market sentiment
3. **Financial Highlights** - Revenue, earnings, key ratios
4. **Risk Assessment** - Key risks, red flags
5. **Forward Looking** - Growth catalysts, guidance

Provide a balanced, objective analysis with specific data points and sources.
""",
    "expected_input": {
        "company": {"type": "rp_entity_id"}
    },
    "research_plan": {
        "title": "Due Diligence Research Plan",
        "steps": [
            {"description": "Gather company overview and business model information"},
            {"description": "Search for recent news and market developments"},
            {"description": "Retrieve financial data and performance metrics"},
            {"description": "Analyze risks from filings and news"},
            {"description": "Synthesize findings into comprehensive assessment"}
        ]
    }
}

print("Template defined:")

response = requests.post(f"{AGENTS_BASE}/v1/workflow/templates", headers=HEADERS, json=due_diligence_template)
response.raise_for_status()

template = response.json()
template_id = template["id"]

print("Template Created Successfully")
print("=" * 50)
print(f"Template ID:     {template_id}")
print(f"Template Name:   {template['name']}")
print(f"Research Steps:  {len(template.get('research_plan', {}).get('steps', []))}")

Execute Template with a company

In [ ]:
# Execute the template
result = run_workflow(
    template=template_id, inputs={"company": "E09E2B"}, # Nvidia
    time_range="last_30_days",
    model="base",
)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result))


In [ ]:
# Execute the template with other company
stripe_result = run_workflow(
    template=template_id, inputs={"company": "98A8F2"}, # Stripe
    time_range="last_30_days",
    model="base",
)

display(Markdown(stripe_result))

## From the app to the API
You can design and save a workflow visually, then call it by ID from your own
systems — the saved template *is* the contract. Explore and build multi-agent
workflows in **Agent Swarm**: 👉 https://agent-swarm.labs.bigdata.com/

---

_Source: [Bigdata.com](https://bigdata.com) · Workflows docs: https://docs.bigdata.com/api-reference/workflows/execute-workflow_